In [1]:
from openai import OpenAI
import json
from dotenv import load_dotenv
from pprint import pprint

_ = load_dotenv()

client = OpenAI()

# 1. Define a list of callable tools for the model

def get_weather(city):
    weather_data = {
        "Paris": "18°C, ensoleillé avec quelques nuages",
        "Berlin": "12°C, pluvieux et nuageux",
        "Lille": "15°C, nuageux avec quelques éclaircies"
    }
    return weather_data.get(city, f"{city}: Données météo non disponibles")

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get weather for",
                },
            },
            "required": ["city"],
        },
    },
]

In [2]:
#Create a running input list we will add to over time
input_list = [
    {"role": "user", "content": "Quel est le temps à Lille aujourd'hui?"}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

pprint(response.output[1])
##Save function call outputs for subsequent requests
input_list += response.output


ResponseFunctionToolCall(arguments='{"city":"Lille"}', call_id='call_4zWmqJ0VFcMBzgIFiWnatzJD', name='get_weather', type='function_call', id='fc_68c3dac6fc608194afbea85a6ac6cf3b0f977718798981e0', status='completed')


In [3]:
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            # 3. Execute the function logic for get_horoscope
            weather = get_weather(**json.loads(item.arguments))
            
            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "get_weather": weather
                })
            })

pprint("Final input:")
pprint(input_list)

'Final input:'
[{'content': "Quel est le temps à Lille aujourd'hui?", 'role': 'user'},
 ResponseReasoningItem(id='rs_68c3dac5f3408194b839e78ca59246d90f977718798981e0', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"city":"Lille"}', call_id='call_4zWmqJ0VFcMBzgIFiWnatzJD', name='get_weather', type='function_call', id='fc_68c3dac6fc608194afbea85a6ac6cf3b0f977718798981e0', status='completed'),
 {'call_id': 'call_4zWmqJ0VFcMBzgIFiWnatzJD',
  'output': '{"get_weather": "15\\u00b0C, nuageux avec quelques '
            '\\u00e9claircies"}',
  'type': 'function_call_output'}]


In [4]:
response = client.responses.create(
    model="gpt-5",
    # instructions="Respond only with the weather generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)


Final output:
{
  "id": "resp_68c3db9b314081949f0c173c22cd088b0f977718798981e0",
  "created_at": 1757666203.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "msg_68c3db9bd53081949048e31c4be84a040f977718798981e0",
      "content": [
        {
          "annotations": [],
          "text": "À Lille aujourd’hui : 15°C, nuageux avec quelques éclaircies.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_weather",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The name of the city to get weather for"
          }
        },
        